In [ ]:
# 导入库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')
# 设置显示选项
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
print("Set completed")
# 加载数据
train_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
print(f"train_data形状: {train_data.shape}")
print(f"test_data形状: {test_data.shape}")

In [ ]:
# 数据信息
print("train data info")
print(train_data.info())
print("\n目标变量统计:")
print(train_data['SalePrice'].describe())
# 变量可视化
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.hist(train_data['SalePrice'], bins=50, alpha=0.7, color='skyblue')
plt.title('SalePrice Distribution')
plt.xlabel('Sale Price')
plt.ylabel('Frequency')
plt.subplot(1, 3, 2)
plt.hist(np.log1p(train_data['SalePrice']), bins=50, alpha=0.7, color='lightgreen')
plt.title('Log(SalePrice) Distribution')
plt.xlabel('Log(Sale Price)')
plt.ylabel('Frequency')
plt.subplot(1, 3, 3)
sns.boxplot(y=train_data['SalePrice'])
plt.title('SalePrice Boxplot')
plt.tight_layout()
plt.show()
# 检查缺失值
missing_train = train_data.isnull().sum().sort_values(ascending=False)
missing_test = test_data.isnull().sum().sort_values(ascending=False)
print("train_data缺失值top 20:")
print(missing_train[missing_train > 0].head(20))
print("\ntest_data缺失值Top 20:")
print(missing_test[missing_test > 0].head(20))

In [ ]:
# 数据信息
print("train data info")
print(train_data.info())
print("\n目标变量统计:")
print(train_data['SalePrice'].describe())
# 变量可视化
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.hist(train_data['SalePrice'], bins=50, alpha=0.7, color='skyblue')
plt.title('SalePrice Distribution')
plt.xlabel('Sale Price')
plt.ylabel('Frequency')
plt.subplot(1, 3, 2)
plt.hist(np.log1p(train_data['SalePrice']), bins=50, alpha=0.7, color='lightgreen')
plt.title('Log(SalePrice) Distribution')
plt.xlabel('Log(Sale Price)')
plt.ylabel('Frequency')
plt.subplot(1, 3, 3)
sns.boxplot(y=train_data['SalePrice'])
plt.title('SalePrice Boxplot')
plt.tight_layout()
plt.show()
# 检查缺失值
missing_train = train_data.isnull().sum().sort_values(ascending=False)
missing_test = test_data.isnull().sum().sort_values(ascending=False)
print("train_data缺失值top 20:")
print(missing_train[missing_train > 0].head(20))
print("\ntest_data缺失值Top 20:")
print(missing_test[missing_test > 0].head(20))

In [ ]:
# 保存ID
train_ids = train_data['Id']
test_ids = test_data['Id']
# 分离特征值和目标变量
y_train = train_data['SalePrice']
X_train = train_data.drop(['Id', 'SalePrice'], axis=1)
X_test = test_data.drop('Id', axis=1)
# 合并数据
all_data = pd.concat([X_train, X_test], axis=0)
print(f"combine_data形状: {all_data.shape}")
# 处理缺失值
def handle_missing_values(df):
    data = df.copy()
    none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
                'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                'MasVnrType']
    for col in none_cols:
        if col in data.columns:
            data[col] = data[col].fillna('None')
    #中位数填充
    numeric_cols = data.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        data[col] = data[col].fillna(data[col].median())
    # 众数填充(列)
    categorical_cols = data.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        data[col] = data[col].fillna(data[col].mode()[0] if len(data[col].mode()) > 0 else 'Unknown')
    return data
all_data_clean = handle_missing_values(all_data)
# 验证缺失值处理
print("处理后缺失值:")
print(all_data_clean.isnull().sum().sum())
# 特征工程 
def create_new_features(df):
    data = df.copy() 
    # 原有的成功特征
    data['TotalSF'] = data['TotalBsmtSF'] + data['1stFlrSF'] + data['2ndFlrSF']
    data['TotalBath'] = data['FullBath'] + 0.5 * data['HalfBath'] + data['BsmtFullBath'] + 0.5 * data['BsmtHalfBath']
    data['TotalPorch'] = data['OpenPorchSF'] + data['EnclosedPorch'] + data['3SsnPorch'] + data['ScreenPorch']
    data['Age'] = data['YrSold'] - data['YearBuilt']
    data['RemodAge'] = data['YrSold'] - data['YearRemodAdd']
    data['IsNew'] = (data['YrSold'] - data['YearBuilt'] <= 1).astype(int)
    data['OverallGrade'] = data['OverallQual'] * data['OverallCond']
    data['KitchenScore'] = data['KitchenQual'].map({'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}) * data['KitchenAbvGr']
    
    # 原有的安全特征
    data['TotalArea'] = data['GrLivArea'] + data['TotalBsmtSF']
    data['BathPerRoom'] = data['TotalBath'] / (data['TotRmsAbvGrd'] + 1)
    data['RoomDensity'] = data['TotRmsAbvGrd'] / (data['GrLivArea'] + 1)
    
    # 经过验证的有效特征
    data['TotalPorchSF'] = data['OpenPorchSF'] + data['EnclosedPorch'] + data['3SsnPorch'] + data['ScreenPorch'] + data['WoodDeckSF']
    data['HasPool'] = (data['PoolArea'] > 0).astype(int)
    data['Has2ndFloor'] = (data['2ndFlrSF'] > 0).astype(int)
    data['HasGarage'] = (data['GarageArea'] > 0).astype(int)
    data['HasFireplace'] = (data['Fireplaces'] > 0).astype(int)
    data['HasBsmt'] = (data['TotalBsmtSF'] > 0).astype(int)
    
    # 豪华住宅标识
    data['LuxuryHome'] = (
        (data['GrLivArea'] > 3500) | (data['OverallQual'] >= 9) |
        (data['GarageCars'] >= 3) | (data['TotalBsmtSF'] > 1500) |
        (data['PoolArea'] > 0) | (data['LotArea'] > 15000)
    ).astype(int)
    
    # NaN值确认
    new_features = ['BathPerRoom', 'RoomDensity', 'LuxuryHome']
    for feature in new_features:
        if feature in data.columns:
            data[feature] = data[feature].fillna(0)
    
    return data

all_data_featured = create_new_features(all_data_clean)

# 编码分类变量
def encode_categorical(df):
    data = df.copy()
    categorical_cols = data.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if col in data.columns:
            le = LabelEncoder()
            try:
                data[col] = le.fit_transform(data[col].astype(str))
            except:
                data[col] = data[col].astype('category').cat.codes
    
    return data

all_data_encoded = encode_categorical(all_data_featured)

# 分割回训练和测试集
X_train_processed = all_data_encoded.iloc[:len(X_train), :]
X_test_processed = all_data_encoded.iloc[len(X_train):, :]

print(f"train_processed形状: {X_train_processed.shape}")
print(f"test_processed形状: {X_test_processed.shape}")

# 数据完整性检查
print("数据完整性检查:")
print(f"X_train_processed NaN数量: {X_train_processed.isnull().sum().sum()}")
print(f"X_test_processed NaN数量: {X_test_processed.isnull().sum().sum()}")

if X_train_processed.isnull().sum().sum() > 0 or X_test_processed.isnull().sum().sum() > 0:
    print("发现NaN值，进行安全填充...")
    
    # 中位数填充
    numeric_cols = X_train_processed.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if col in X_train_processed.columns:
            train_median = X_train_processed[col].median()
            X_train_processed[col] = X_train_processed[col].fillna(train_median)
            X_test_processed[col] = X_test_processed[col].fillna(train_median)
    
    print("填充后数据完整性检查:")
    print(f"X_train_processed NaN数量: {X_train_processed.isnull().sum().sum()}")
    print(f"X_test_processed NaN数量: {X_test_processed.isnull().sum().sum()}")

X_train_selected = X_train_processed
X_test_selected = X_test_processed
print(f"使用所有特征进行训练，特征数量: {X_train_selected.shape[1]}")

#分割回训练和测试集
X_train_processed = all_data_encoded.iloc[:len(X_train), :]
X_test_processed = all_data_encoded.iloc[len(X_train):, :]

print(f"train_processed形状: {X_train_processed.shape}")
print(f"test_processed形状: {X_test_processed.shape}")

In [ ]:
# Forest
#rf_for_feature_importance = RandomForestRegressor(n_estimators=100, random_state=42)
#rf_for_feature_importance.fit(X_train_processed, y_train)
X_train_selected = X_train_processed
X_test_selected = X_test_processed
print(f"使用所有特征进行训练，特征数量: {X_train_selected.shape[1]}")
y_train_log = np.log1p(y_train)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_selected, y_train_log, test_size=0.2, random_state=42
)
#获取特征重要性
#feature_importance = pd.DataFrame({
    #'feature': X_train_processed.columns,
    #'importance': rf_for_feature_importance.feature_importances_
#}).sort_values('importance', ascending=False)
#尝试多个模型
models = {
    'RandomForest': RandomForestRegressor(n_estimators=700, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=1200,  
        random_state=42,
        learning_rate=0.015,  
        max_depth=4,          
        min_samples_split=30, 
        min_samples_leaf=15,  
        subsample=0.8,        
        max_features='sqrt'   
    ),
    'Lasso': Lasso(alpha=0.0005, random_state=42, max_iter=2000),  # 调整alpha
    'Ridge': Ridge(alpha=10.0, random_state=42)  # 增加正则化强度
}
results = {}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    results[name] = rmse
    print(f"{name} RMSE: {rmse:.4f}")
best_model_name = min(results, key=results.get)
print(f"\nbest_model: {best_model_name}")
# 训练最佳模型
if best_model_name == 'RandomForest':
    best_model = RandomForestRegressor(n_estimators=700, random_state=42)
elif best_model_name == 'GradientBoosting': best_model = GradientBoostingRegressor(
   n_estimators=1200,  
        random_state=42,
        learning_rate=0.015,  
        max_depth=4,          
        min_samples_split=30, 
        min_samples_leaf=15,  
        subsample=0.8,        # 增加子采样
        max_features='sqrt'   
)
elif best_model_name == 'Lasso':
    best_model = Lasso(alpha=0.0005, random_state=42, max_iter=2000)
else:
    best_model = Ridge(alpha=10.0, random_state=42)

best_model.fit(X_train_selected, y_train_log)

print("\n开始集成学习...")

# 选择表现最好的几个模型进行集成
models_to_ensemble = {
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=1200,  
        random_state=42,
        learning_rate=0.015,  
        max_depth=4,          
        min_samples_split=30, 
        min_samples_leaf=15,  
        subsample=0.8,        # 增加子采样
        max_features='sqrt'   
    ),
    'RandomForest': RandomForestRegressor(
        n_estimators=300,  # 减少树数量以加快训练
        random_state=42,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5
    ),
    'Lasso': Lasso(alpha=0.0005, random_state=42, max_iter=2000)  # 调整alpha
}

# 训练所有集成模型
ensemble_predictions_val = {}
ensemble_models = {}

for name, model in models_to_ensemble.items():
    print(f"训练集成模型: {name}")
    model.fit(X_tr, y_tr)
    ensemble_models[name] = model
    # 在验证集上预测
    y_pred_val = model.predict(X_val)
    ensemble_predictions_val[name] = y_pred_val
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    print(f"{name} 验证集RMSE: {rmse:.4f}")

# 计算每个模型的权重
weights = {}
for name, pred in ensemble_predictions_val.items():
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    # RMSE越小，权重越大
    weights[name] = 1 / (rmse + 1e-6)  

# 归一化权重
total_weight = sum(weights.values())
for name in weights:
    weights[name] /= total_weight

print("\n模型权重:")
for name, weight in weights.items():
    print(f"{name}: {weight:.4f}")

# 在验证集上测试集成效果
ensemble_val_pred = np.zeros_like(y_val)
for name, pred in ensemble_predictions_val.items():
    ensemble_val_pred += weights[name] * pred

ensemble_rmse = np.sqrt(mean_squared_error(y_val, ensemble_val_pred))
print(f"集成模型验证集RMSE: {ensemble_rmse:.4f}")

# 比较集成模型和最佳单个模型
if ensemble_rmse < results[best_model_name]:
    print("集成模型表现更好，使用集成模型")
    
    # 在整个训练集上重新训练集成模型
    for name, model in models_to_ensemble.items():
        print(f"在整个训练集上重新训练: {name}")
        model.fit(X_train_selected, y_train_log)
        ensemble_models[name] = model
    
    # 对测试集进行集成预测
    test_predictions_log = np.zeros(X_test_selected.shape[0])
    for name, model in ensemble_models.items():
        pred = model.predict(X_test_selected)
        test_predictions_log += weights[name] * pred
    
else:
    print("最佳单个模型表现更好，使用最佳单个模型")

def smart_post_process(predictions, y_train):
    """
    智能后处理：基于训练集分布调整预测值
    """
    predictions_clean = predictions.copy()
    
    # 计算训练集的统计量
    train_q1 = np.percentile(y_train, 25)
    train_q3 = np.percentile(y_train, 75)
    train_iqr = train_q3 - train_q1
    lower_bound = train_q1 - 1.5 * train_iqr
    upper_bound = train_q3 + 1.5 * train_iqr
    
    # 修剪极端异常值
    predictions_clean = np.clip(predictions_clean, lower_bound, upper_bound)
    
    # 四舍五入到百位（借鉴房地产商）
    predictions_clean = np.round(predictions_clean / 100) * 100
    
    print(f"后处理范围: [{lower_bound:.0f}, {upper_bound:.0f}]")
    print(f"后处理前统计: 均值={predictions.mean():.0f}, 标准差={predictions.std():.0f}")
    print(f"后处理后统计: 均值={predictions_clean.mean():.0f}, 标准差={predictions_clean.std():.0f}")
    
    return predictions_clean


In [ ]:
test_predictions_log = best_model.predict(X_test_selected)
test_predictions = np.expm1(test_predictions_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_predictions
})
submission.to_csv('submission.csv', index=False)

print("提交文件已生成!")